In [ ]:
import os
import sys
from pathlib import Path

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"

HADOOP_HOME = "/opt/hadoop-3.3.6"

os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["HADOOP_CONF_DIR"] = f"{HADOOP_HOME}/etc/hadoop"
os.environ["YARN_CONF_DIR"] = f"{HADOOP_HOME}/etc/hadoop"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["SPARK_LOCAL_IP"] = "10.0.2.15"

METASTORE = BASE / "metastore"
METASTORE.mkdir(parents=True, exist_ok=True)

print("Proyecto:", BASE)
print("Metastore:", METASTORE)
print("Python:", sys.executable)

In [ ]:
from pyspark.sql import SparkSession

derby_path = (METASTORE / "metastore_db").as_posix()

spark = (
    SparkSession.builder
    .appName("afp_v2_hive_catalogo")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.driver.host", "10.0.2.15")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config("spark.sql.shuffle.partitions", "4")
    .config(
        "javax.jdo.option.ConnectionURL",
        f"jdbc:derby:;databaseName={derby_path};create=true"
    )
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        f"jdbc:derby:;databaseName={derby_path};create=true"
    )
    .config(
        "spark.sql.warehouse.dir",
        "hdfs://localhost:9000/user/junior/datalake_afp_v2/catalog/warehouse"
    )
    .enableHiveSupport()
    .getOrCreate()
)

print("Spark:", spark.version)
print("MASTER:", spark.sparkContext.master)
print("CATALOGO:", spark.conf.get("spark.sql.catalogImplementation"))

spark.sql("SHOW DATABASES").show()

In [ ]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS afp_analytics
""")

spark.sql("SHOW DATABASES").show()

In [ ]:
BASE_HDFS = (
    "hdfs://localhost:9000"
    "/user/junior/datalake_afp_v2/processed"
)

tablas = [
    "bcrp_series_largas",
    "bcrp_json_rentabilidad",
    "sbs_afiliados",
    "sbs_rentabilidad",
    "sbs_valor_cuota",
    "memorias_indicadores",
    "tabla_analitica_mensual"
]

for nombre in tablas:

    ruta = f"{BASE_HDFS}/{nombre}"

    df = spark.read.parquet(ruta)

    df.createOrReplaceTempView(
        f"tmp_{nombre}"
    )

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS
        afp_analytics.{nombre}
        USING PARQUET
        LOCATION '{ruta}'
    """)

    print("Tabla registrada:", nombre)

print("\nTablas en el catálogo:")
spark.sql(
    "SHOW TABLES IN afp_analytics"
).show(truncate=False)

In [ ]:
for nombre in tablas:

    resultado = spark.sql(
        f"SELECT COUNT(*) AS total "
        f"FROM afp_analytics.{nombre}"
    ).collect()[0]["total"]

    print(f"{nombre}: {resultado} registros")

In [ ]:
tablas_particionadas = [
    "bcrp_series_largas",
    "bcrp_json_rentabilidad",
    "sbs_afiliados",
    "sbs_rentabilidad",
    "sbs_valor_cuota",
    "memorias_indicadores"
]

for nombre in tablas_particionadas:
    tabla = f"afp_analytics.{nombre}"

    print(f"\nRecuperando particiones de: {tabla}")

    spark.sql(
        f"MSCK REPAIR TABLE {tabla}"
    )

    print("Particiones recuperadas correctamente")

In [ ]:
for nombre in tablas:
    total = spark.sql(
        f"SELECT COUNT(*) AS total "
        f"FROM afp_analytics.{nombre}"
    ).first()["total"]

    print(f"{nombre}: {total} registros")

In [ ]:
spark.sql("""
SHOW PARTITIONS afp_analytics.bcrp_series_largas
""").show(10, truncate=False)

In [ ]:
spark.stop()
print("Sesión Spark cerrada correctamente")